In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dev_catalog;
CREATE SCHEMA IF NOT EXISTS dev_catalog.bronze;

CREATE SCHEMA IF NOT EXISTS dev_catalog.gold
MANAGED LOCATION 'abfss://gold@mamataustrafficstorage.dfs.core.windows.net/' ;

CREATE SCHEMA IF NOT EXISTS dev_catalog.silver
MANAGED LOCATION 'abfss://silver@mamataustrafficstorage.dfs.core.windows.net/' ;

CREATE SCHEMA IF NOT EXISTS dev_catalog.landing;
CREATE SCHEMA IF NOT EXISTS dev_catalog.checkpoints;

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS landing
URL 'abfss://landing@mamataustrafficstorage.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL ustrafficcredential);

CREATE EXTERNAL LOCATION IF NOT EXISTS bronze
URL 'abfss://bronze@mamataustrafficstorage.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL ustrafficcredential);

CREATE EXTERNAL LOCATION IF NOT EXISTS silver
URL 'abfss://silver@mamataustrafficstorage.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL ustrafficcredential);

CREATE EXTERNAL LOCATION IF NOT EXISTS gold
URL 'abfss://gold@mamataustrafficstorage.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL ustrafficcredential);

CREATE EXTERNAL LOCATION IF NOT EXISTS checkpoints
URL 'abfss://checkpoints@mamataustrafficstorage.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL ustrafficcredential);

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS dev_catalog.landing.landing_vol
LOCATION 'abfss://landing@mamataustrafficstorage.dfs.core.windows.net/';

CREATE EXTERNAL VOLUME IF NOT EXISTS dev_catalog.checkpoints.checkpoint_vol
LOCATION 'abfss://checkpoints@mamataustrafficstorage.dfs.core.windows.net/'



In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_catalog.bronze.raw_roads (
    count_point_id STRING,
    year STRING,
    region_id STRING,
    region_name STRING,
    region_ons_code STRING,
    local_authority_id STRING,
    local_authority_name STRING,
    local_authority_code STRING,
    road_name STRING,
    road_category STRING,
    road_type STRING,
    start_junction_road_name STRING,
    end_junction_road_name STRING,
    easting STRING,
    northing STRING,
    latitude STRING,
    longitude STRING,
    link_length_km STRING,
    link_length_miles STRING,
    source STRING,
    Extract_Time TIMESTAMP
    
)
USING DELTA
LOCATION 'abfss://bronze@mamataustrafficstorage.dfs.core.windows.net/raw_roads';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_catalog.bronze.raw_traffic(
    count_point_id STRING,
    year STRING,
    region_id STRING,
    region_name STRING,
    region_ons_code STRING,
    local_authority_id STRING,
    local_authority_name STRING,
    local_authority_code STRING,
    road_name STRING,
    road_category STRING,
    road_type STRING,
    start_junction_road_name STRING,
    end_junction_road_name STRING,
    easting STRING,
    northing STRING,
    latitude STRING,
    longitude STRING,
    estimation_method STRING,
    estimation_method_detailed STRING,
    direction_of_travel STRING,
    pedal_cycles STRING,
    two_wheeled_motor_vehicles STRING,
    cars_and_taxis STRING,
    buses_and_coaches STRING,
    LGVs STRING,
    HGVs_2_rigid_axle STRING,
    HGVs_3_rigid_axle STRING,
    HGVs_4_or_more_rigid_axle STRING,
    HGVs_3_or_4_articulated_axle STRING,
    HGVs_5_articulated_axle STRING,
    HGVs_6_articulated_axle STRING,
    all_HGVs STRING,
    source STRING,
    link_length_km STRING,
    link_length_miles STRING,
    all_motor_vehicles STRING,
    Extract_Time TIMESTAMP
)
USING DELTA
location 'abfss://bronze@mamataustrafficstorage.dfs.core.windows.net/raw_traffic';


In [0]:
try:
    spark.sql("""
           CREATE TABLE IF NOT EXISTS dev_catalog.bronze.vehicle_type_lookup (
            vehicle_code STRING,
            vehicle_label STRING
            );""")
    
    spark.sql("""
            INSERT INTO dev_catalog.bronze.vehicle_type_lookup VALUES
            ('EV_Car', 'Electric Car'), ('EV_Bike', 'Electric Bike'),
            ('LGV_Type', 'Light Goods Vehicle'), ('HGV_Type', 'Heavy Goods Vehicle')
        """)
except Exception as e:
    log_pipeline_error("create_table_raw_traffic",e)
    raise

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_catalog.silver.roads (
    count_point_id INT NOT NULL,
    road_type_consistent BOOLEAN,
    year INT NOT NULL,
    region_id STRING,
    region_name STRING,
    region_ons_code STRING,
    local_authority_id STRING,
    local_authority_name STRING,
    local_authority_code STRING,
    road_name STRING,
    road_category STRING,
    road_type STRING,
    start_junction_road_name STRING,
    end_junction_road_name STRING,
    easting STRING,
    northing STRING,
    latitude STRING,
    longitude STRING,
    link_length_km DOUBLE,
    link_length_miles DOUBLE,
    valid_from TIMESTAMP,
    valid_to TIMESTAMP,
    source STRING,
    transformed_Time TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_catalog.silver.traffic (
    count_point_id INT NOT NULL,
    year INT NOT NULL,
    direction_of_travel STRING NOT NULL,
    estimation_method STRING,
    is_estimated BOOLEAN,
    link_length_km DOUBLE,
    pedal_cycles INT,
    two_wheeled_motor_vehicles INT,
    cars_and_taxis INT,
    buses_and_coaches INT,
    LGVs INT,
    HGVs_2_rigid_axle INT,
    HGVs_3_rigid_axle INT,
    HGVs_4_or_more_rigid_axle INT,
    HGVs_3_or_4_articulated_axle INT,
    HGVs_5_articulated_axle INT,
    HGVs_6_articulated_axle INT,
    all_HGVs INT,
    all_motor_vehicles INT,
    recomputed_motor_vehicles INT,
    vehicle_count_consistent BOOLEAN,
    total_traffic_volume INT,
    vehicle_intensity DOUBLE,
    transformed_time TIMESTAMP,
    link_length_miles DOUBLE,
    source STRING
)
USING DELTA
CLUSTER BY AUTO;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_catalog.default.pipeline_errors
(
    notebook STRING, 
    step STRING,
    error_message STRING,
    error_time TIMESTAMP
)

In [0]:
%sql
SELECT COUNT(*) from dev_catalog.bronze.raw_roads;

In [0]:
%sql
SELECT COUNT(*) from dev_catalog.silver.roads;

In [0]:
%sql
SELECT * from dev_catalog.bronze.raw_roads
WHERE `_rescued_data` IS NOT NULL;

In [0]:
%sql
select * from dev_catalog.default.pipeline_errors;